# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [12]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [13]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [14]:
links = fetch_website_links("https://edwarddonner.com")
links

/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'edwarddonner.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [15]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [16]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
print(get_links_user_prompt("https://edwarddonner.com"))

/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'edwarddonner.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(



Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/

In [20]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links


In [21]:
select_relevant_links("https://edwarddonner.com")

/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'edwarddonner.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [22]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [23]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'edwarddonner.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 10 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'skills page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'profile page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'partner site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [24]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 11 relevant links


{'links': [{'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'forum page', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'community page', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'product page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [25]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [26]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Selecting relevant links for https://huggingface.co by calling gpt-5-nano


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 20 relevant links


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
nvidia/LocateAnything-3B
Updated
3 days ago
•
132k
•
1.86k
google/gemma-4-12B-it
Updated
7 days ago
•
676k
•
934
google/diffusiongemma-26B-A4B-it
Updated
1 day ago
•
449
bosonai/higgs-audio-v3-tts-4b
Updated
about 23 hours ago
•
19.9k
•
349
CohereLabs/North-Mini-Code-1.0
Updated
about 9 hours ago
•
1.86k
•
301
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
190
TripoSplat
👁
190
Ge

In [27]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [28]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [29]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Selecting relevant links for https://huggingface.co by calling gpt-5-nano


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 11 relevant links


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apply.workable.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to h

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nnvidia/LocateAnything-3B\nUpdated\n3 days ago\n•\n132k\n•\n1.86k\ngoogle/gemma-4-12B-it\nUpdated\n7 days ago\n•\n676k\n•\n934\

In [30]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [32]:
create_brochure("HuggingFace", "https://huggingface.co")

/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 

# Hugging Face Company Brochure

---

## About Hugging Face

**Hugging Face** is the premier AI community dedicated to building the future of machine learning. It serves as an open collaboration platform where developers, researchers, and enterprises come together to create, share, and deploy state-of-the-art models, datasets, and applications. Hugging Face empowers millions globally by providing access to over **2 million models**, **500,000 datasets**, and **1 million+ AI applications**.

---

## Our Platform

- **Models**: Browse, host, and collaborate on a vast collection of AI models spanning numerous tasks such as natural language processing, computer vision, and audio processing.
- **Datasets**: A rich repository of datasets backed by community contributions and institutional partners enabling innovative AI training.
- **Spaces**: Deploy machine learning applications and demos in easy-to-use interactive spaces without extensive setup.
- **Buckets**: Scalable storage solutions for machine learning workflows.
- **HuggingChat**: Cutting-edge chat and conversational AI tooling built for seamless interaction.
- **Enterprise Solutions**: Robust Pro and Enterprise offerings including dedicated support, inference providers, scalable endpoints, and secure storage options designed for business-critical deployments.

---

## Community & Collaboration

Hugging Face thrives on its vibrant community-driven ecosystem including:

- Active developer forums and **Discord** channels fostering knowledge exchange and support.
- Public repositories on **GitHub** allowing open collaboration and innovation.
- Daily blogs and research paper highlights that keep the community informed on the latest advances.
- Partnerships with leading institutions such as NVIDIA, Google, Stanford Vision Lab, and more who contribute key models and datasets.

---

## Customers & Use Cases

Hugging Face is trusted by AI professionals, researchers, startups, and large enterprises alike, spanning domains like:

- Computer vision (object detection, video analysis)
- Natural Language Processing (chatbots, text generation, translation)
- Audio and speech applications (text-to-speech, voice recognition)
- AI-powered search and recommendation systems
- Scientific research and academic exploration

Notable trending resources on the platform include models like `nvidia/LocateAnything-3B`, `google/gemma-4-12B-it`, and large-scale datasets curated by prominent labs.

---

## Career Opportunities

Join Hugging Face and be part of the AI revolution! The company fosters a culture of openness, innovation, and community impact. Working here means contributing to:

- Open-source machine learning projects used worldwide.
- Cutting-edge AI research with real-world applications.
- A collaborative environment that values diversity and encourages continuous learning.

Explore opportunities across engineering, research, data science, product, and enterprise teams to shape the future of AI.

---

## Brand & Identity

- **Colors:** Vibrant yellow (#FFD21E), orange (#FF9D00), and gray (#6B7280).
- Hugging Face brand assets including logos and design materials are freely available for use in projects promoting open collaboration.

---

## Join Us

Be a part of the community that’s building tomorrow’s AI today. Discover, collaborate, and innovate on the Hugging Face platform:

- Visit: [huggingface.co](https://huggingface.co)
- Engage via Discord and Forums
- Contribute on GitHub
- Explore, share, and deploy models in minutes

Hugging Face is where the machine learning community grows together.

---

*Hugging Face: The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [3]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [34]:
openai = OpenAI()
stream_brochure("HuggingFace", "https://huggingface.co")


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Selecting relevant links for https://huggingface.co by calling gpt-5-nano


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Found 12 relevant links


/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apply.workable.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'huggingface.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/jude.pieries/agentic-ai/agenetic_engineeting/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to h

# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a pioneering AI company and platform dedicated to building the future through an open, collaborative machine learning community. It empowers researchers, developers, and enterprises by providing access to millions of AI models, datasets, and applications all in one place. The company’s platform fosters innovation by enabling anyone to discover, create, and collaborate on machine learning models and tools.

---

## What They Offer

- **Models:** Over 2 million machine learning models available for a wide range of AI tasks including natural language processing, computer vision, and audio processing.  
- **Datasets:** Access to more than 500,000 datasets to train, test, and deploy models.  
- **Spaces:** Collaborative environments for AI applications where users can experiment and deploy projects easily.  
- **Enterprise Solutions:** Specialized support, inference endpoints, and storage buckets designed for business needs.  
- **HuggingChat:** An AI chat interface that exemplifies the company’s commitment to innovative conversational AI.

---

## Community & Collaboration

Hugging Face is much more than a platform—it is a vibrant community hub. It hosts a thriving network where AI enthusiasts and professionals contribute to open-source models, share knowledge through blogs, forums, and Discord channels, and advance AI research collaboratively. With daily updates on cutting-edge papers, case studies, and tools, Hugging Face keeps its community deeply engaged and informed.

---

## Notable Customers & Partners

Hugging Face serves a diverse user base from individual researchers to large enterprises. Key partnerships include leading technology organizations like NVIDIA and Google. Enterprises leverage Hugging Face’s infrastructure to power advanced AI use cases, replace legacy systems, and accelerate AI-driven transformation with multi-million dollar commercial partnerships.

---

## Company Culture

- **Open and Inclusive:** Embracing open-source collaboration and accessibility for all.  
- **Innovative and Research-Driven:** Focus on state-of-the-art AI models and staying at the forefront of emerging technology.  
- **Community-Focused:** Strong emphasis on user, developer, and researcher engagement.  
- **Ethical AI Advocacy:** Commitment to ethical practices in AI development and deployment.  

Hugging Face fosters a creative environment where innovation thrives through collective intelligence and shared passion for AI.

---

## Careers at Hugging Face

Hugging Face is continuously expanding and seeking talented individuals passionate about AI, machine learning, and open-source technology. Current openings span engineering, research, product management, and community roles. The company offers an inspiring work environment with opportunities to contribute to cutting-edge AI projects and make a real-world impact.

Explore available positions and join a fast-growing team dedicated to shaping the future of AI.

---

## Get Involved

- **Explore 2M+ models and 500k+ datasets** on the Hugging Face website.  
- **Participate in community discussions** via the Hugging Face Discord, forum, and GitHub repositories.  
- **Read cutting-edge AI research and case studies** in their active blog.  
- **Try HuggingChat** and other AI-powered applications built on Hugging Face technology.  

---

## Contact & Links

- Website: [huggingface.co](https://huggingface.co)  
- Careers: Check the [Careers page](https://huggingface.co/careers) for job openings  
- Community & Support: Join the [Discord](https://discord.gg/huggingface), forum, and GitHub  
- Blog: Latest AI insights and stories [here](https://huggingface.co/blog)  

---

Hugging Face empowers the AI community to innovate collaboratively, driving forward the future of machine learning for everyone. Join the revolution today!

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>